In [ ]:
using Plots
using LinearAlgebra
using Revise
using May12Project

# Neumann Boundary Value Problem

In [ ]:
a = 0.0;
b = 1.0;
nx = 20;
N = nx + 1; # number points, including endpoints

x = LinRange(a, b, N); # include endpoints
@show Δx = x[2] - x[1];

Dxx = sparse_second_derivative_matrix(N, Δx);
Dxx

In [ ]:
# edit Dxx[1,1] and Dxx[end,end] to enforce Neumann BCs
Dxx[1,1] = -1/Δx^2;
Dxx[end,end] = -1/Δx^2;
Dxx

In [ ]:
A = -Dxx;
f =x-> π^2 * cos(π* x);
rhs = f.(x);
u = A\rhs;

In [ ]:
eigvals(Matrix(A))

In [ ]:
cond(Matrix(A))

**NOTE** The zero eigenvalue, under floating point arithemtic, turns into something that's machine precision, and *not* an genuine zero.

In [ ]:
plot(x, u, marker=:circle, label="FD Solution")
plot!(x, cos.(π* x), label="Exact Solution")
xlabel!("x")

## Check Convergence

In [ ]:
nx_values = [5, 10, 20, 40, 80, 160, 320, 640, 1280, 2560, 5120, 10240, 20480, 40960, 81920, 163840, 327680, 655360, 1310720];
N_values = nx_values .+ 1;
errors = zeros(length(nx_values));
for (i, N) in enumerate(N_values)
    x = LinRange(a, b, N);
    Δx = x[2] - x[1];
    Dxx = sparse_second_derivative_matrix(N, Δx);
    Dxx[1,1] = -1/Δx^2;
    Dxx[end,end] = -1/Δx^2;
    A = -Dxx;
    rhs = f.(x);
    u = A\rhs;
    errors[i] =  norm(u .- cos.(π* x), Inf);
end

In [ ]:
scatter(1 ./nx_values, errors, xscale=:log10, yscale=:log10, marker=:circle, label="Errors")
plot!( 1 ./ nx_values, 10 * (1 ./ nx_values), label="O(h)", linestyle=:dash)
plot!( 1 ./ nx_values, 10 * (1 ./ nx_values.^2), label="O(h^2)", linestyle=:dash, legend=:bottomright)
xlabel!("h")

In contrast to the Dirichlet problme, where we saw $\mathrm{O}(h^2)$ error, here we see non-monotonic $\mathrm{O}(h)$ error.

This reflects that in this implementation, we made a $\mathrm{O}(h)= \mathrm{O}(\delta x)$ approxiamtion of the Neumann boundary conditions:
$$
0= u'(0)= \frac{u_0 - u_{-1}}{\delta x} + \mathrm{O}(\delta x)
$$

## Quadratic Approximation

In [ ]:
a = 0.0;
b = 1.0;
nx = 5;
N = nx + 1; # number points, including endpoints

x = LinRange(a, b, N); # include endpoints
@show Δx = x[2] - x[1];

Dxx = sparse_second_derivative_matrix(N, Δx);

# edit Dxx[1,1] and Dxx[end,end] to enforce Neumann BCs
Dxx[1,2] = 2/Δx^2;
Dxx[end,end-1] = 2/Δx^2;
Dxx

A = -Dxx;
f =x-> π^2 * cos(π* x);
rhs = f.(x);


In [ ]:
A

In [ ]:
eigvals(Matrix(A))

In [ ]:
cond(Matrix(A))

In [ ]:
u = A\rhs;

In [ ]:
u = qr(A)\rhs; # (solved it)

In [ ]:
plot(x, u, marker=:circle, label="FD Solution")
plot!(x, cos.(π* x), label="Exact Solution")
xlabel!("x")

Vertical offset that reflects the null space.

In [ ]:
plot(x, u .-u[1], marker=:circle, label="FD Solution")
plot!(x, cos.(π* x) .- cos(π *x[1]), label="Exact Solution")
xlabel!("x")

## Check convergence

In [ ]:
nx_values = [5, 10, 20, 40, 80, 160, 320, 640, 1280, 2560, 5120, 10240, 20480, 40960, 81920, 163840, 327680, 655360, 1310720];
N_values = nx_values .+ 1;
errors = zeros(length(nx_values));
for (i, N) in enumerate(N_values)
    x = LinRange(a, b, N);
    Δx = x[2] - x[1];
    Dxx = sparse_second_derivative_matrix(N, Δx);
    Dxx[1,2] = 2/Δx^2;
    Dxx[end,end-1] = 2/Δx^2;
    A = -Dxx;
    rhs = f.(x);
    u = qr(A)\rhs;
    errors[i] =  norm( (u.-u[1]) .-(cos.(π* x) .- cos(π *x[1])), Inf);
end

In [ ]:
scatter(1 ./nx_values, errors, xscale=:log10, yscale=:log10, marker=:circle, label="Errors")
plot!( 1 ./ nx_values, 10 * (1 ./ nx_values), label="O(h)", linestyle=:dash)
plot!( 1 ./ nx_values, 10 * (1 ./ nx_values.^2), label="O(h^2)", linestyle=:dash, legend=:bottomright)
xlabel!("h")

Up till the minimum, it is quadratic error, $\mathrm{O}(\delta x^2)$.  The U shape is due to floating point error.